# 25 — Consensus modellerinin tüm dış test setleriyle değerlendirilmesi

Bu notebook 24 numaralı notebook'a dokunmaz ve onun sonuçlarını yeniden yazmaz.
Consensus ile eğitilmiş dört final modeli, 2024 ana deneylerinde kullanılan tam dış test kümelerinde değerlendirir:

- S&P 500: 1.060 örnek
- Sentetik finansal haber: 3.000 örnek
- Reuters: 5.000 örnek

Önemli: Burada eğitimdeki consensus filtresi dış testlere uygulanmaz. Dış test etiketleri, mevcut tam dış test dosyalarından alınır.

In [1]:
from pathlib import Path
import gc, inspect, json, re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report, confusion_matrix
from thesis_utils import DATA_ROOT, paths

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABELS = list(LABEL2ID)
MAX_LENGTH, BATCH_SIZE = 128, 64
MODEL_ROOT = paths.MODEL_CHECKPOINT_ROOT / 'consensus_twitter_phrasebank_allagree_v1'
RESULTS_ROOT = paths.MODEL_RESULTS_ROOT / 'consensus_twitter_phrasebank_allagree_v1' / 'evaluation_all_external'
SP500_DIR = paths.SP500_HUMAN_REVIEW_BATCHES_DIR
SYNTHETIC_DIR = paths.SYNTHETIC_NEWS_DIR
REUTERS_FULL = paths.REUTERS_ANNOTATION_DIR / 'REUTERS_annotation_all_clean.csv'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CONFIGS = [
    ('bert', 'BERT-base-uncased'),
    ('distilbert', 'DistilBERT-base-uncased'),
    ('roberta', 'RoBERTa-base'),
    ('finbert', 'FinBERT'),
]

def norm_cols(df):
    out = df.copy()
    out.columns = [re.sub(r'[^a-z0-9]+', '_', str(c).strip().lower()).strip('_') for c in out.columns]
    return out

def clean_value(s):
    return s.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'none': pd.NA, '<na>': pd.NA})

def read_excel_batch(path):
    raw = pd.read_excel(path, header=None, dtype=object)
    header = None
    for i in range(min(20, len(raw))):
        vals = {str(x).strip().lower() for x in raw.iloc[i].tolist()}
        if 'annotation_id' in vals or ('label' in vals and ('text' in vals or 'text_en' in vals)):
            header = i; break
    if header is None: header = 0
    df = raw.iloc[header + 1:].copy(); df.columns = raw.iloc[header].tolist()
    return df.dropna(how='all').dropna(axis=1, how='all')

def make_eval(df, name, label_cols):
    df = norm_cols(df)
    text_col = next((c for c in ['input_text','text_en','text','source_text','headline','title'] if c in df.columns), None)
    if text_col is None: raise ValueError(f'{name}: text kolonu bulunamadı: {df.columns.tolist()}')
    df['eval_text'] = clean_value(df[text_col])
    df['gold_label'] = pd.NA
    for c in label_cols:
        if c in df.columns: df['gold_label'] = df['gold_label'].fillna(clean_value(df[c]).str.lower())
    df = df[df['eval_text'].notna() & df['gold_label'].isin(LABELS)].copy().reset_index(drop=True)
    df['gold_id'] = df['gold_label'].map(LABEL2ID).astype(int)
    if 'sample_id' not in df.columns: df.insert(0, 'sample_id', [f'{name}_{i:06d}' for i in range(len(df))])
    return df

def load_sp500():
    files = sorted(SP500_DIR.glob('SP500_annotation_batch_*.xlsx'))
    if not files: raise FileNotFoundError(SP500_DIR)
    return make_eval(pd.concat([read_excel_batch(p) for p in files], ignore_index=True), 'sp500_external', ['final_label','chatgpt_label'])

def load_synthetic():
    files = sorted(SYNTHETIC_DIR.glob('*_batch_*.xlsx'))
    if not files: raise FileNotFoundError(SYNTHETIC_DIR)
    return make_eval(pd.concat([read_excel_batch(p) for p in files], ignore_index=True), 'synthetic_external', ['label'])

def load_reuters():
    if not REUTERS_FULL.exists(): raise FileNotFoundError(REUTERS_FULL)
    return make_eval(pd.read_csv(REUTERS_FULL), 'reuters_external', ['final_label','chatgpt_label'])

eval_sets = {
    'sp500_external': load_sp500(),
    'synthetic_external': load_synthetic(),
    'reuters_external': load_reuters(),
}
print({k: {'n': len(v), 'labels': v['gold_label'].value_counts().to_dict()} for k, v in eval_sets.items()})
assert len(eval_sets['sp500_external']) == 1060, len(eval_sets['sp500_external'])
assert len(eval_sets['synthetic_external']) == 3000, len(eval_sets['synthetic_external'])
assert len(eval_sets['reuters_external']) == 5000, len(eval_sets['reuters_external'])

{'sp500_external': {'n': 1060, 'labels': {'positive': 398, 'neutral': 339, 'negative': 323}}, 'synthetic_external': {'n': 3000, 'labels': {'negative': 1000, 'neutral': 1000, 'positive': 1000}}, 'reuters_external': {'n': 5000, 'labels': {'positive': 2275, 'negative': 1579, 'neutral': 1146}}}


In [2]:
def predict(model_key, df):
    model_dir = MODEL_ROOT / model_key / 'final_model'
    if not model_dir.exists(): raise FileNotFoundError(model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to('cuda' if torch.cuda.is_available() else 'cpu')
    device = next(model.parameters()).device; model.eval(); rows = []
    texts = df['eval_text'].astype(str).tolist()
    with torch.no_grad():
        for start in tqdm(range(0, len(texts), BATCH_SIZE), desc=f'{model_key}'):
            enc = tokenizer(texts[start:start+BATCH_SIZE], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
            enc = {k: v.to(device) for k, v in enc.items()}
            probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
            rows.append(probs)
    probs = np.concatenate(rows); pred = probs.argmax(axis=1)
    return pred, probs

def evaluate(model_key, display_name, dataset_name, df):
    out = RESULTS_ROOT / model_key / dataset_name; out.mkdir(parents=True, exist_ok=True)
    pred, probs = predict(model_key, df); y = df['gold_id'].to_numpy()
    precision, recall, f1, support = precision_recall_fscore_support(y, pred, labels=[0,1,2], zero_division=0)
    metrics = {'model_key': model_key, 'model': display_name, 'dataset': dataset_name, 'n_eval': len(df), 'accuracy': accuracy_score(y,pred), 'precision_macro': precision.mean(), 'recall_macro': recall.mean(), 'f1_macro': f1.mean(), 'f1_weighted': f1_score(y,pred,average='weighted'), 'f1_negative': f1[0], 'f1_neutral': f1[1], 'f1_positive': f1[2], 'support_negative': int(support[0]), 'support_neutral': int(support[1]), 'support_positive': int(support[2])}
    pred_df = df.copy(); pred_df['predicted_label'] = [ID2LABEL[int(x)] for x in pred]; pred_df['confidence'] = probs.max(axis=1)
    for i, label in ID2LABEL.items(): pred_df[f'prob_{label}'] = probs[:,i]
    pred_df.to_csv(out / 'predictions.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(classification_report(y,pred,labels=[0,1,2],target_names=LABELS,output_dict=True,zero_division=0)).T.to_csv(out / 'classification_report.csv', encoding='utf-8-sig')
    cm = confusion_matrix(y,pred,labels=[0,1,2]); pd.DataFrame(cm,index=[f'true_{x}' for x in LABELS],columns=[f'pred_{x}' for x in LABELS]).to_csv(out / 'confusion_matrix.csv', encoding='utf-8-sig')
    (out / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    return metrics

all_results = []
for model_key, display_name in MODEL_CONFIGS:
    for dataset_name, df in eval_sets.items():
        print(f'Evaluate: {display_name} | {dataset_name} | n={len(df)}')
        all_results.append(evaluate(model_key, display_name, dataset_name, df))
        gc.collect()
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(RESULTS_ROOT / 'all_models_all_external_summary.csv', index=False, encoding='utf-8-sig')
display(summary_df[['model','dataset','n_eval','accuracy','f1_macro','f1_weighted']].round(4))

Evaluate: BERT-base-uncased | sp500_external | n=1060


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

bert:   0%|          | 0/17 [00:00<?, ?it/s]

Evaluate: BERT-base-uncased | synthetic_external | n=3000


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

bert:   0%|          | 0/47 [00:00<?, ?it/s]

... (27 outputs left out)

Evaluate: FinBERT | reuters_external | n=5000


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

finbert:   0%|          | 0/79 [00:00<?, ?it/s]

,model,dataset,n_eval,accuracy,f1_macro,f1_weighted
0,BERT-base-uncased,sp500_external,1060,0.8075,0.8066,0.8074
1,BERT-base-uncased,synthetic_external,3000,0.9817,0.9816,0.9816
2,BERT-base-uncased,reuters_external,5000,0.6622,0.6663,0.6791
3,DistilBERT-base-uncased,sp500_external,1060,0.7811,0.7797,0.7808
4,DistilBERT-base-uncased,synthetic_external,3000,0.9687,0.9686,0.9686
5,DistilBERT-base-uncased,reuters_external,5000,0.6380,0.6426,0.6551
6,RoBERTa-base,sp500_external,1060,0.8330,0.8319,0.8328
7,RoBERTa-base,synthetic_external,3000,0.9957,0.9957,0.9957
8,RoBERTa-base,reuters_external,5000,0.6690,0.6769,0.6894
9,FinBERT,sp500_external,1060,0.8066,0.8069,0.8075
